In [0]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

# ADLS
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
container_name = os.getenv("CONTAINER_NAME")

# SQL Server
sql_host = os.getenv("SQL_HOST")
sql_database = os.getenv("SQL_DATABASE")
sql_username = os.getenv("SQL_USERNAME")
sql_password = os.getenv("SQL_PASSWORD")

print("Variáveis carregadas com sucesso!")

In [0]:
base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"

path_rastreamento = base_path + "vendas_raw/2026/02/21/112200/ecommerce_rastreamento.parquet"

print(path_rastreamento)

In [0]:
df_rastreamento = (
    spark.read
    .format("parquet")
    .option(
        f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net",
        "OAuth"
    )
    .option(
        f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
    )
    .option(
        f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net",
        client_id
    )
    .option(
        f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net",
        client_secret
    )
    .option(
        f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net",
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
    )
    .load(path_rastreamento)
)

display(df_rastreamento)

In [0]:
df_rastreamento.printSchema()

In [0]:
jdbc_url = (
    f"jdbc:sqlserver://{sql_host}:1433;"
    f"database={sql_database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

print(jdbc_url)

In [0]:
(
    df_rastreamento.write
    .format("sqlserver")
    .option("host", sql_host)
    .option("port", "1433")
    .option("database", sql_database)
    .option("dbtable", "ecommerce_rastreamento")
    .option("user", sql_username)
    .option("password", sql_password)
    .mode("overwrite")
    .save()
)

print("Tabela ecommerce_rastreamento criada com sucesso!")